In [ ]:
import os
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [ ]:
class TrafficDataset(Dataset):
    def __init__(self, csv_path, image_dir, transform=None):
        self.data = pd.read_csv(csv_path)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_name = self.data.iloc[idx]["image_name"]
        y = float(self.data.iloc[idx]["congestion_score"])

        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(y, dtype=torch.float32)


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [ ]:
import torch.nn as nn
from torchvision import models


In [ ]:
class CongestionCNN(nn.Module):
    def __init__(self):
        super().__init__()

        backbone = models.resnet18(pretrained=True)
        self.feature_extractor = nn.Sequential(*list(backbone.children())[:-1])

        self.regressor = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        x = x.view(x.size(0), -1)
        x = self.regressor(x)
        return x.squeeze(1)


In [ ]:
model = CongestionCNN()

for param in model.feature_extractor.parameters():
    param.requires_grad = False


In [ ]:
criterion = nn.SmoothL1Loss()

optimizer = torch.optim.Adam(
    model.regressor.parameters(),
    lr=1e-3
)


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


In [ ]:
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            targets = targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            total_loss += loss.item()

    return total_loss / len(loader)


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

for epoch in range(25):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss = validate(model, val_loader, criterion, device)

    print(f"Epoch {epoch+1}: train={train_loss:.4f}, val={val_loss:.4f}")


In [ ]:
for param in model.feature_extractor[-2:].parameters():
    param.requires_grad = True


In [ ]:
optimizer = torch.optim.Adam([
    {"params": model.feature_extractor.parameters(), "lr": 1e-5},
    {"params": model.regressor.parameters(), "lr": 1e-4}
])


In [ ]:
for epoch in range(15):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss = validate(model, val_loader, criterion, device)

    print(f"Fine-tune {epoch+1}: train={train_loss:.4f}, val={val_loss:.4f}")


In [ ]:
model.eval()

with torch.no_grad():
    img, _ = dataset[0]
    pred = model(img.unsqueeze(0).to(device))
    print(float(pred))
